# Single File Weather Data Cleaning

**Project**: FLAPS  
**Dataset**: A **single** daily weather data from BOM in csv formal  
**Purpose**: Clean weather data, rename columns for clarity and convert from csv to dataframe. Only to check individual csv file. Bulk data cleaning does not perform data validation.

---

## Table of Contents
1. [Setup](#1-setup)
2. [Load data](#2-load-data)
3. [Initial inspection](#3-initial-inspection)

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

## 2. Load data

Read the .csv data and rename the variable (column) names.

In [11]:
# Skipping the last row because it contains the total value only for some columns
df = pd.read_csv('../data/raw/syd/sydney_airport_amo-202512.csv', skiprows=range(1,13), skipfooter=1, encoding='latin-1', engine='python')

# Replace column titles
df.columns = ['station', 'date', 'evapotranspiration', 'rain', 'panevaporation', 'temp_max', 'temp_min', 'hum_max', 'hum_min', 'wind_speed', 'solar_rad']

# Remove station column as it's not needed (same value for all rows)
df = df.drop('station', axis=1)

print(f"Data loaded successfully!")
print(f"Columns: {list(df.columns)}")

Data loaded successfully!
Columns: ['date', 'evapotranspiration', 'rain', 'panevaporation', 'temp_max', 'temp_min', 'hum_max', 'hum_min', 'wind_speed', 'solar_rad']


## 3. Initial inspection
High level overview of the data frame.

In [12]:
# View the first few entries
print(df.head())

         date evapotranspiration  rain panevaporation  temp_max  temp_min  \
0  01/12/2025                8.1   0.0           12.6      26.5      15.5   
1  02/12/2025                6.4   0.0            9.4      21.1      13.4   
2  03/12/2025                7.6   0.0            8.8      26.8      14.2   
3  04/12/2025                9.4   0.0            8.4      32.5      17.7   
4  05/12/2025               11.7   0.0           13.2      37.8      19.6   

   hum_max  hum_min  wind_speed solar_rad  
0       55       19        7.36     21.90  
1       66       43        8.10     31.22  
2       71       27        5.51     31.46  
3       77       19        5.98     31.45  
4       66        8        6.26     30.79  


In [4]:
# Basic statistics of the variables
df.describe()

,rain,temp_max,temp_min,hum_max,hum_min,wind_speed
count,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000
mean,0.690323,28.280645,18.329032,80.870968,38.258065,6.023871
std,1.454958,6.004966,2.641362,9.749331,16.905557,1.258752
min,0.000000,18.900000,13.400000,55.000000,8.000000,3.950000
25%,0.000000,23.950000,16.500000,77.000000,24.500000,4.975000
50%,0.000000,26.500000,19.000000,82.000000,41.000000,6.070000
75%,0.400000,32.650000,19.650000,87.000000,50.000000,7.010000
max,5.800000,42.600000,23.000000,95.000000,77.000000,8.100000


## 4. Check for Missing Values
Identify any missing or null values in the dataset.

In [5]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\n" + "="*50 + "\n")

# Get more detailed info
print("Dataset info:")
df.info()

Missing values per column:
station               0
date                  0
evapotranspiration    0
rain                  0
panevaporation        0
temp_max              0
temp_min              0
hum_max               0
hum_min               0
wind_speed            0
solar_rad             0
dtype: int64


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   station             31 non-null     object 
 1   date                31 non-null     object 
 2   evapotranspiration  31 non-null     object 
 3   rain                31 non-null     float64
 4   panevaporation      31 non-null     object 
 5   temp_max            31 non-null     float64
 6   temp_min            31 non-null     float64
 7   hum_max             31 non-null     int64  
 8   hum_min             31 non-null     int64  
 9   wind_speed          31 non-null  

## 5. Data Type Conversion
Convert columns to appropriate data types.

In [6]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

# Verify the conversion
print("Date column after conversion:")
print(df['date'].head())
print(f"\nData type: {df['date'].dtype}")

Date column after conversion:
0   2025-12-01
1   2025-12-02
2   2025-12-03
3   2025-12-04
4   2025-12-05
Name: date, dtype: datetime64[ns]

Data type: datetime64[ns]


## 6. Check for Duplicates
Identify any duplicate date entries in the dataset.

In [ ]:
# Check for duplicate dates
duplicates = df[df.duplicated(subset=['date'], keep=False)]

print(f"Number of duplicate date entries: {len(duplicates)}")

if len(duplicates) > 0:
    print("\nDuplicate rows:")
    print(duplicates.sort_values('date'))
else:
    print("No duplicate dates found - dataset is clean!")

## 7. Data Validation
Check for anomalies and values outside expected ranges.

In [ ]:
# Check for negative values in columns that shouldn't have them
print("Checking for invalid negative values:")
print("="*60)

numeric_cols = ['evapotranspiration', 'rain', 'panevaporation', 'wind_speed', 'solar_rad']
for col in numeric_cols:
    # Convert to numeric first in case there are string values
    df[col] = pd.to_numeric(df[col], errors='coerce')
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"⚠ {col}: {negative_count} negative values found")
    else:
        print(f"✓ {col}: All values valid (non-negative)")

# Check temperature consistency
print("\n" + "="*60)
print("Temperature consistency check:")
temp_issues = df[df['temp_max'] < df['temp_min']]
if len(temp_issues) > 0:
    print(f"⚠ Found {len(temp_issues)} records where temp_max < temp_min:")
    print(temp_issues[['date', 'temp_max', 'temp_min']])
else:
    print("✓ All temperatures consistent (temp_max >= temp_min)")

# Check humidity ranges (should be 0-100)
print("\n" + "="*60)
print("Humidity range check:")
hum_issues = df[(df['hum_max'] > 100) | (df['hum_max'] < 0) | (df['hum_min'] > 100) | (df['hum_min'] < 0)]
if len(hum_issues) > 0:
    print(f"⚠ Found {len(hum_issues)} records with humidity outside 0-100% range")
else:
    print("✓ All humidity values within valid range (0-100%)")

## 8. Cleaned Dataset Summary
Final overview of the cleaned and validated dataset.

In [ ]:
# Display cleaned dataset summary
print("="*60)
print("CLEANED DATASET SUMMARY")
print("="*60)
print(f"\n📊 Total records: {len(df)}")
print(f"📅 Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"🔢 Total columns: {len(df.columns)}")
print(f"❌ Total missing values: {df.isnull().sum().sum()}")

print("\n" + "="*60)
print("Data Types:")
print(df.dtypes)

print("\n" + "="*60)
print("First 5 rows of cleaned data:")
print(df.head())